In [9]:
_ARTIFACT_PROMPT = """Analyze this architectural image and extract structured artifacts as JSON.

Return exactly this JSON shape (no extra keys, no markdown):
{
  "title": "one concise sentence — style, program, key feature (under 15 words)",
  "description": "2-3 sentences: building character, notable visual qualities, architectural intent",
  "building_type": "<residential|cultural|religious|commercial|civic|institutional|industrial|infrastructure|landscape>",
  "style": {
    "primary": "<style_label>",
    "secondary": [],
    "confidence": 0.0,
    "style_evidence": [],
    "emergent_tags": []
  },
  "architectural_elements": {
    "structural": [],
    "facade": [],
    "roofing": [],
    "openings": [],
    "ornamental": [],
    "circulation": []
  },
  "materials": [],
  "material_details": {
    "textures": [],
    "construction_expression": []
  },
  "spatial_features": {
    "massing": [],
    "geometry": [],
    "symmetry": [],
    "rhythm": [],
    "depth_layering": []
  },
  "color_palette": {
    "dominant": [],
    "accent": [],
    "material_tones": []
  },
  "environment": {
    "setting": [],
    "urban_context": [],
    "landscape": [],
    "climate_indicators": []
  },
  "viewpoint": {
    "camera_angle": "",
    "view_type": "",
    "facade_visibility": ""
  },
  "relationships": [
    {"source": "<element>", "relation": "<relation_type>", "target": "<element>"}
  ],
  "semantic_keywords": [],
  "retrieval_tags": []
}

Rules:
- Use normalized architectural ontology tags only (lowercase_underscored labels)
- No prose, no explanations outside the JSON values
- description: 2-3 prose sentences about the building's character and visual identity
- style_evidence: list of visual features that support the primary style classification
- emergent_tags: unique or rare architectural features not covered by canonical style labels
- confidence: float 0.0-1.0 based on visual clarity
- materials: flat list of primary material names (e.g. reinforced_concrete, glass_curtain_wall)
- retrieval_tags: 5-10 concise tags optimized for semantic search and discovery
- relationships: only when a structural dependency is clearly visible
- Output valid JSON only
"""


In [16]:
# 1. Set your API key
OLLAMA_API_KEY = "0d989aa8f5eb4eba83839d536fa2357b.ZpNAAsrR-ytKsYnG3t5vG7Z2"  # Or set in env var


In [26]:
import os
import base64
from ollama import Client

client = Client(
    host="https://ollama.com",
    headers={"Authorization": "Bearer " + OLLAMA_API_KEY}
)

# 2. Load an image and encode it to base64
with open("test-data/AncientEgyptianArchitecture.webp", "rb") as image_file:
    base64_image = base64.standard_b64encode(image_file.read()).decode('utf-8')

# 3. Send request to a vision-capable cloud model
response = client.chat(
    model="gemma4:31b-cloud",  # Use a fully vision-capable model
    messages=[{
        "role": "user",
        "content": _ARTIFACT_PROMPT,
        "images": [base64_image]   # Crucial: Include images as base64[reference:8]
    }]
)

# 4. Print the response
print(response["message"]["content"])

{
  "title": "Ancient Egyptian temple complex featuring monumental sculptures and pyramidal forms",
  "description": "A massive stone complex characterized by colossal seated figures and heavy colonnades. The architecture emphasizes permanence and divine scale, integrated directly into a rugged cliffside landscape.",
  "building_type": "religious",
  "style": {
    "primary": "ancient_egyptian",
    "secondary": [],
    "confidence": 1.0,
    "style_evidence": [
      "colossal_statuary",
      "hypostyle_columns",
      "pyramidal_forms",
      "hieroglyphic_scale"
    ],
    "emergent_tags": [
      "monolithic_carving",
      "cliff_integrated"
    ]
  },
  "architectural_elements": {
    "structural": [
      "post_and_lintel",
      "massive_plinths"
    ],
    "facade": [
      "pylon_gateway",
      "colonnade"
    ],
    "roofing": [
      "flat_roof"
    ],
    "openings": [
      "rectangular_portals"
    ],
    "ornamental": [
      "anthropomorphic_sculpture",
      "sphinx

In [27]:
import os, base64, io
from PIL import Image
from ollama import Client

client = Client(host="https://ollama.com", headers={"Authorization": "Bearer " + OLLAMA_API_KEY})

def optimize_image(image_path, max_size=1024, quality=85):
    with Image.open(image_path) as img:
        if img.mode in ('RGBA', 'P'): img = img.convert('RGB')
        ratio = max_size / max(img.size)
        if ratio < 1:
            new_size = (int(img.size[0] * ratio), int(img.size[1] * ratio))
            img = img.resize(new_size, Image.Resampling.LANCZOS)
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=quality, optimize=True)
        return base64.standard_b64encode(buffer.getvalue()).decode('utf-8')

base64_image = optimize_image("test-data/AncientEgyptianArchitecture.webp")

response = client.chat(
    model="gemma4:31b-cloud",
    messages=[{"role": "user", "content": _ARTIFACT_PROMPT, "images": [base64_image]}]
)
print(response["message"]["content"])

```json
{
  "title": "Ancient Egyptian monumental complex featuring pylons, columns, and colossal sculptures",
  "description": "A vast ceremonial complex characterized by massive limestone structures and colossal figurative sculpture. The architecture emphasizes permanence and power through heavy massing and rhythmic colonnades set against a rugged desert cliffside.",
  "building_type": "religious",
  "style": {
    "primary": "ancient_egyptian",
    "secondary": [],
    "confidence": 1.0,
    "style_evidence": [
      "hypostyle_hall",
      "colossal_statues",
      "pylon_gates",
      "pyramidal_forms",
      "hieroglyphic_reliefs"
    ],
    "emergent_tags": [
      "monolithic_scale",
      "theocratic_urbanism"
    ]
  },
  "architectural_elements": {
    "structural": [
      "post_and_lintel",
      "massive_walls",
      "retaining_walls"
    ],
    "facade": [
      "colonnade",
      "pylon_face",
      "carved_reliefs"
    ],
    "roofing": [
      "flat_roof",
      "sla